# NIS Database Analysis

This notebook provides a template for analyzing the National Inpatient Sample (NIS) data.

**Database Stats:**
- 56.3 million hospitalizations (2013-2020)
- 8 years of data
- Unified `nis_core` view + year-specific tables

In [ ]:
# Setup
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Connect to database
con = duckdb.connect('/Volumes/Niels 1/NIS/NIS.duckdb', read_only=True)
print("Connected to NIS database")

## Quick Reference: Key Columns

| Column | Description |
|--------|-------------|
| `AGE` | Patient age in years |
| `FEMALE` | 1=Female, 0=Male |
| `DIED` | 1=Died during hospitalization |
| `LOS` | Length of stay (days) |
| `TOTCHG` | Total charges (USD) |
| `PAY1` | Primary payer (1=Medicare, 2=Medicaid, 3=Private, 4=Self-pay) |
| `RACE` | 1=White, 2=Black, 3=Hispanic, 4=Asian/Pacific, 5=Native American |
| `HOSP_REGION` | 1=Northeast, 2=Midwest, 3=South, 4=West |
| `YEAR` | Discharge year |
| `I10_DX1` | Primary diagnosis (ICD-10, 2016+) |
| `DX1` | Primary diagnosis (ICD-9, 2013-2014) |

## 1. Overview: Trends Over Time

In [ ]:
# Annual trends
df_trends = con.execute("""
    SELECT 
        YEAR,
        COUNT(*) as admissions,
        SUM(DIED) as deaths,
        ROUND(100.0 * SUM(DIED) / COUNT(*), 2) as mortality_pct,
        ROUND(AVG(LOS), 2) as avg_los,
        ROUND(AVG(TOTCHG), 0) as avg_charges
    FROM nis_core
    GROUP BY YEAR
    ORDER BY YEAR
""").df()

df_trends

In [ ]:
# Visualize trends
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Admissions
axes[0, 0].bar(df_trends['YEAR'], df_trends['admissions']/1e6, color='steelblue')
axes[0, 0].set_title('Annual Admissions (Millions)')
axes[0, 0].set_ylabel('Admissions (M)')

# Mortality rate
axes[0, 1].plot(df_trends['YEAR'], df_trends['mortality_pct'], marker='o', color='crimson', linewidth=2)
axes[0, 1].set_title('In-Hospital Mortality Rate (%)')
axes[0, 1].set_ylabel('Mortality %')

# Average LOS
axes[1, 0].bar(df_trends['YEAR'], df_trends['avg_los'], color='teal')
axes[1, 0].set_title('Average Length of Stay (Days)')
axes[1, 0].set_ylabel('Days')

# Average charges
axes[1, 1].bar(df_trends['YEAR'], df_trends['avg_charges']/1000, color='orange')
axes[1, 1].set_title('Average Total Charges ($K)')
axes[1, 1].set_ylabel('Charges ($K)')

plt.tight_layout()
plt.show()

## 2. Demographics Analysis

In [ ]:
# Age distribution
df_age = con.execute("""
    SELECT 
        CASE 
            WHEN AGE < 1 THEN '0 (Newborn)'
            WHEN AGE BETWEEN 1 AND 17 THEN '1-17'
            WHEN AGE BETWEEN 18 AND 44 THEN '18-44'
            WHEN AGE BETWEEN 45 AND 64 THEN '45-64'
            WHEN AGE BETWEEN 65 AND 84 THEN '65-84'
            ELSE '85+'
        END as age_group,
        COUNT(*) as admissions,
        ROUND(100.0 * SUM(DIED) / COUNT(*), 2) as mortality_pct,
        ROUND(AVG(LOS), 1) as avg_los,
        ROUND(AVG(TOTCHG), 0) as avg_charges
    FROM nis_core
    WHERE AGE IS NOT NULL
    GROUP BY 1
    ORDER BY 
        CASE 
            WHEN age_group = '0 (Newborn)' THEN 1
            WHEN age_group = '1-17' THEN 2
            WHEN age_group = '18-44' THEN 3
            WHEN age_group = '45-64' THEN 4
            WHEN age_group = '65-84' THEN 5
            ELSE 6
        END
""").df()

df_age

In [ ]:
# Payer distribution
df_payer = con.execute("""
    SELECT 
        CASE PAY1
            WHEN 1 THEN 'Medicare'
            WHEN 2 THEN 'Medicaid'
            WHEN 3 THEN 'Private Insurance'
            WHEN 4 THEN 'Self-Pay'
            WHEN 5 THEN 'No Charge'
            WHEN 6 THEN 'Other'
            ELSE 'Unknown'
        END as payer,
        COUNT(*) as admissions,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) as pct_total,
        ROUND(100.0 * SUM(DIED) / COUNT(*), 2) as mortality_pct
    FROM nis_core
    GROUP BY PAY1
    ORDER BY admissions DESC
""").df()

# Pie chart
fig, ax = plt.subplots(figsize=(10, 8))
colors = sns.color_palette('Set2', len(df_payer))
ax.pie(df_payer['admissions'], labels=df_payer['payer'], autopct='%1.1f%%', colors=colors)
ax.set_title('Distribution by Primary Payer (2013-2020)')
plt.show()

df_payer

## 3. Regional Analysis

In [ ]:
# By region
df_region = con.execute("""
    SELECT 
        CASE HOSP_REGION
            WHEN 1 THEN 'Northeast'
            WHEN 2 THEN 'Midwest'
            WHEN 3 THEN 'South'
            WHEN 4 THEN 'West'
        END as region,
        COUNT(*) as admissions,
        ROUND(100.0 * SUM(DIED) / COUNT(*), 2) as mortality_pct,
        ROUND(AVG(LOS), 1) as avg_los,
        ROUND(AVG(TOTCHG), 0) as avg_charges,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY TOTCHG), 0) as median_charges
    FROM nis_core
    WHERE HOSP_REGION IS NOT NULL AND TOTCHG IS NOT NULL
    GROUP BY HOSP_REGION
    ORDER BY HOSP_REGION
""").df()

df_region

## 4. COVID-19 Analysis (2020)

In [ ]:
# COVID-19 cases in 2020
df_covid = con.execute("""
    SELECT 
        CASE 
            WHEN I10_DX1 LIKE 'U07%' THEN 'COVID Primary'
            WHEN I10_DX2 LIKE 'U07%' OR I10_DX3 LIKE 'U07%' OR 
                 I10_DX4 LIKE 'U07%' OR I10_DX5 LIKE 'U07%' THEN 'COVID Secondary'
            ELSE 'Non-COVID'
        END as covid_status,
        COUNT(*) as cases,
        SUM(DIED) as deaths,
        ROUND(100.0 * SUM(DIED) / COUNT(*), 2) as mortality_pct,
        ROUND(AVG(LOS), 1) as avg_los,
        ROUND(AVG(TOTCHG), 0) as avg_charges
    FROM nis_2020
    GROUP BY 1
    ORDER BY cases DESC
""").df()

df_covid

In [ ]:
# COVID mortality by age
df_covid_age = con.execute("""
    SELECT 
        CASE 
            WHEN AGE < 45 THEN '0-44'
            WHEN AGE < 65 THEN '45-64'
            WHEN AGE < 75 THEN '65-74'
            WHEN AGE < 85 THEN '75-84'
            ELSE '85+'
        END as age_group,
        COUNT(*) as covid_cases,
        SUM(DIED) as deaths,
        ROUND(100.0 * SUM(DIED) / COUNT(*), 1) as mortality_pct
    FROM nis_2020
    WHERE I10_DX1 LIKE 'U07%' OR I10_DX2 LIKE 'U07%' OR I10_DX3 LIKE 'U07%'
    GROUP BY 1
    ORDER BY 1
""").df()

# Bar plot
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(df_covid_age['age_group'], df_covid_age['mortality_pct'], color='indianred')
ax.set_title('COVID-19 In-Hospital Mortality by Age Group (2020)')
ax.set_xlabel('Age Group')
ax.set_ylabel('Mortality Rate (%)')
for bar, val in zip(bars, df_covid_age['mortality_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val}%', ha='center')
plt.show()

df_covid_age

## 5. Top Diagnoses Analysis

In [ ]:
# Top 20 diagnoses in 2020
df_dx = con.execute("""
    SELECT 
        I10_DX1 as icd10_code,
        COUNT(*) as cases,
        ROUND(100.0 * SUM(DIED) / COUNT(*), 2) as mortality_pct,
        ROUND(AVG(LOS), 1) as avg_los,
        ROUND(AVG(TOTCHG), 0) as avg_charges
    FROM nis_2020
    WHERE I10_DX1 IS NOT NULL
    GROUP BY I10_DX1
    ORDER BY cases DESC
    LIMIT 20
""").df()

df_dx

## 6. Custom Query Template

Use this cell to run your own queries:

In [ ]:
# Custom query - modify as needed
query = """
    SELECT *
    FROM nis_core
    WHERE YEAR = 2020
    LIMIT 10
"""

df_custom = con.execute(query).df()
df_custom

In [ ]:
# Close connection when done
# con.close()